# Wan 2.2 T2V-A14B Video Generation on AWS Trainium2

End-to-end text-to-video generation with **Wan 2.2 T2V-A14B** (27B MoE, 14B active per step) on a single `trn2.48xlarge` instance.

## Architecture

- **Model**: Wan 2.2 T2V-A14B — Mixture of Experts (MoE) with 2 transformer experts (14.29B params each, zero weight sharing)
- **Resolution**: 768×1280, 81 frames, 40 denoising steps
- **Parallelism**: TP=4, CP=16 (64 NeuronCores per expert)
- **Key optimizations**:
  - Subprocess isolation for clean HBM lifecycle (separate process per expert)
  - Context Parallelism (CP=16) splits sequence across all 64 cores for near-linear scaling
  - NKI Flash Attention (attention_isa_kernel)
  - Neuron tiled VAE decode (8 parallel tiles on 8 NCs)

## Pipeline Flow

```
1. Text encoding with Neuron-compiled UMT5-XXL (subprocess)
2. Expert 1 denoising on 16 cores (13 high-noise steps)
3. Expert 2 denoising on 16 cores (27 low-noise steps)
4. Neuron tiled VAE decode (8 tiles on 8 NCs)
5. Export video
```

## Prerequisites

- `trn2.48xlarge` instance with Deep Learning AMI Neuron (Ubuntu 24.04) 20260502 (SDK 2.29.1)
- Pre-installed venv: `/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference/`
- NVMe storage mounted (1.7 TB)

**Expected total time**: ~30 min compile + ~20 min inference = ~50 min first run

## Step 1: Environment Setup

In [ ]:
import os
import subprocess
import sys

# Configuration
NVME_MOUNT = "/opt/dlami/nvme"
MODELS_DIR = f"{NVME_MOUNT}/models"
MODEL_NAME = "Wan2.2-T2V-A14B-Diffusers"
MODEL_DIR = f"{MODELS_DIR}/{MODEL_NAME}"
COMPILED_DIR = f"{NVME_MOUNT}/compiled_a14b_tp4"
COMPILER_WORKDIR = f"{NVME_MOUNT}/compiler_workdir"
CACHE_DIR = f"{NVME_MOUNT}/wan2.2_t2v_a14b_hf_cache_dir"
HENAN_DIR = f"{NVME_MOUNT}/aws-neuron-samples/torch-neuronx/inference/hf_pretrained_wan2.2_t2v_a14b"
VENV = "/opt/aws_neuronx_venv_pytorch_2_9_nxd_inference"

# Parallelism
TP_DEGREE = 4
CP_DEGREE = 16
WORLD_SIZE = TP_DEGREE * CP_DEGREE  # 64

# Video generation
HEIGHT = 768
WIDTH = 1280
NUM_FRAMES = 81
NUM_STEPS = 40

print(f"TP={TP_DEGREE}, CP={CP_DEGREE}, World Size={WORLD_SIZE}")
print(f"Resolution: {HEIGHT}x{WIDTH}, {NUM_FRAMES} frames, {NUM_STEPS} steps")

In [ ]:
# Verify Neuron devices
!neuron-ls

In [ ]:
# Verify SDK versions
!pip show neuronx-cc torch-neuronx neuronx-distributed 2>/dev/null | grep -E 'Name:|Version:'

## Step 2: Mount NVMe Storage

In [ ]:
%%bash -e
NVME_MOUNT="/opt/dlami/nvme"

if mountpoint -q "${NVME_MOUNT}" 2>/dev/null; then
    echo "NVMe already mounted at ${NVME_MOUNT}"
else
    # Find NVMe devices (exclude root EBS)
    NVME_DEVICES=$(lsblk -d -n -o NAME,TYPE | grep disk | grep nvme | awk '{print "/dev/"$1}' | sort)
    ROOT_DEV=$(findmnt -n -o SOURCE / | sed 's/p[0-9]*$//' | sed 's/[0-9]*$//')

    NVME_TO_MOUNT=""
    for dev in $NVME_DEVICES; do
        if [[ "$dev" != "$ROOT_DEV"* ]] && ! lsblk -n -o MOUNTPOINT "$dev" | grep -q '/'; then
            NVME_TO_MOUNT="$dev"
            break
        fi
    done

    if [ -z "$NVME_TO_MOUNT" ]; then
        echo "No unmounted NVMe found, using EBS storage"
        sudo mkdir -p "${NVME_MOUNT}"
        sudo chown ubuntu:ubuntu "${NVME_MOUNT}"
    else
        echo "Formatting and mounting ${NVME_TO_MOUNT}..."
        sudo mkfs.ext4 -F "$NVME_TO_MOUNT"
        sudo mkdir -p "${NVME_MOUNT}"
        sudo mount "$NVME_TO_MOUNT" "${NVME_MOUNT}"
        sudo chown ubuntu:ubuntu "${NVME_MOUNT}"
        echo "Mounted at ${NVME_MOUNT}"
    fi
fi

df -h ${NVME_MOUNT}

In [ ]:
# Create directories
for d in [MODELS_DIR, COMPILED_DIR, COMPILER_WORKDIR, CACHE_DIR]:
    os.makedirs(d, exist_ok=True)
print("Directories created.")

## Step 3: Install Dependencies

In [ ]:
# Install required packages
!pip install -q diffusers==0.38.0 accelerate imageio-ffmpeg


In [ ]:
# Patch diffusers: nearest-exact -> nearest (required for Trainium2 VAE)
import diffusers

vae_path = os.path.join(
    os.path.dirname(diffusers.__file__),
    "models", "autoencoders", "autoencoder_kl_wan.py"
)
with open(vae_path) as f:
    content = f.read()
if "nearest-exact" in content:
    content = content.replace("nearest-exact", "nearest")
    with open(vae_path, "w") as f:
        f.write(content)
    print("Patched diffusers: nearest-exact -> nearest")
else:
    print("Diffusers already patched")

## Step 4: Clone Neuron Samples (Compilation Code)

In [ ]:
%%bash
NVME_MOUNT="/opt/dlami/nvme"
SAMPLES_DIR="${NVME_MOUNT}/aws-neuron-samples"
HENAN_DIR="${SAMPLES_DIR}/torch-neuronx/inference/hf_pretrained_wan2.2_t2v_a14b"

if [ -d "${HENAN_DIR}" ]; then
    echo "aws-neuron-samples already exists"
else
    echo "Cloning aws-neuron-samples..."
    git clone --depth 1 https://github.com/whn09/aws-neuron-samples.git "${SAMPLES_DIR}"
fi

echo "Compilation code directory:"
ls ${HENAN_DIR}/neuron_wan2_2_t2v_a14b/ || echo 'ERROR: Directory not found'

## Step 5: Download Model (118 GB)

This downloads the full Wan 2.2 T2V-A14B model from HuggingFace (~10-15 minutes).

In [ ]:
if os.path.exists(os.path.join(MODEL_DIR, "model_index.json")):
    print(f"Model already downloaded at {MODEL_DIR}")
else:
    print(f"Downloading Wan-AI/Wan2.2-T2V-A14B-Diffusers (118 GB)...")
    from huggingface_hub import snapshot_download
    snapshot_download(
        "Wan-AI/Wan2.2-T2V-A14B-Diffusers",
        local_dir=MODEL_DIR,
        local_dir_use_symlinks=False,
    )
    print("Download complete.")

!du -sh {MODEL_DIR}

## Step 6: Compile Models

Compilation of all 4 artifacts:
1. **Text encoder** (UMT5-XXL) — ~2 min
2. **Transformer Expert 1** (high-noise) — ~5 min
3. **Transformer Expert 2** (low-noise) — ~5 min
4. **Tiled VAE decoder** (416×416 tiles) — ~3 min

Total compilation: ~15 minutes.

In [ ]:
# Set up environment for compilation
os.environ["NEURON_RT_VIRTUAL_CORE_SIZE"] = "2"
os.environ["PYTHONPATH"] = HENAN_DIR + ":" + os.environ.get("PYTHONPATH", "")
print(f"PYTHONPATH includes: {HENAN_DIR}")

In [ ]:
%%bash -e
# Cache HuggingFace model
NVME_MOUNT="/opt/dlami/nvme"
HENAN_DIR="${NVME_MOUNT}/aws-neuron-samples/torch-neuronx/inference/hf_pretrained_wan2.2_t2v_a14b"
CACHE_DIR="${NVME_MOUNT}/wan2.2_t2v_a14b_hf_cache_dir"

export PYTHONPATH="${HENAN_DIR}:${PYTHONPATH:-}"
python ${HENAN_DIR}/neuron_wan2_2_t2v_a14b/cache_hf_model.py --cache_dir ${CACHE_DIR} 2>&1 | tail -5

### 6a: Compile Text Encoder

In [ ]:
%%bash -e
NVME_MOUNT="/opt/dlami/nvme"
HENAN_DIR="${NVME_MOUNT}/aws-neuron-samples/torch-neuronx/inference/hf_pretrained_wan2.2_t2v_a14b"
COMPILED_DIR="${NVME_MOUNT}/compiled_a14b_tp4"
CACHE_DIR="${NVME_MOUNT}/wan2.2_t2v_a14b_hf_cache_dir"
export PYTHONPATH="${HENAN_DIR}:${PYTHONPATH:-}"
export NEURON_RT_VIRTUAL_CORE_SIZE=2

if [ -f "${COMPILED_DIR}/text_encoder/nxd_model.pt" ]; then
    echo "Text encoder already compiled"
else
    echo "Compiling text encoder (TP=4)..."
    WORLD_SIZE=8 neuron_parallel_compile python \
        ${HENAN_DIR}/neuron_wan2_2_t2v_a14b/compile_text_encoder.py \
        --max_sequence_length 512 \
        --tp_degree 4 --world_size 8 \
        --compiled_models_dir ${COMPILED_DIR} \
        --cache_dir ${CACHE_DIR} \
        2>&1 | tail -10
    echo "Text encoder compiled."
fi

ls -lh ${COMPILED_DIR}/text_encoder/nxd_model.pt

### 6b: Compile Transformer Expert 1 (High Noise)

This is the larger compilation step (~5 minutes). The transformer has 40 layers with NKI flash attention.

In [ ]:
%%bash -e
NVME_MOUNT="/opt/dlami/nvme"
HENAN_DIR="${NVME_MOUNT}/aws-neuron-samples/torch-neuronx/inference/hf_pretrained_wan2.2_t2v_a14b"
COMPILED_DIR="${NVME_MOUNT}/compiled_a14b_tp4"
CACHE_DIR="${NVME_MOUNT}/wan2.2_t2v_a14b_hf_cache_dir"
COMPILER_WORKDIR="${NVME_MOUNT}/compiler_workdir"
export PYTHONPATH="${HENAN_DIR}:${PYTHONPATH:-}"
export NEURON_RT_VIRTUAL_CORE_SIZE=2

if [ -f "${COMPILED_DIR}/transformer/nxd_model.pt" ]; then
    echo "Expert 1 already compiled"
else
    echo "Compiling transformer Expert 1 (TP=4, CP=16)..."
    WORLD_SIZE=64 neuron_parallel_compile python \
        ${HENAN_DIR}/neuron_wan2_2_t2v_a14b/compile_transformer.py \
        --height 768 --width 1280 --num_frames 81 --max_sequence_length 512 \
        --tp_degree 4 --cp_degree 16 --batch_size 1 \
        --transformer_subfolder transformer \
        --compiled_models_dir ${COMPILED_DIR} \
        --compiler_workdir ${COMPILER_WORKDIR}/expert1 \
        --cache_dir ${CACHE_DIR} \
        2>&1 | tail -10
    echo "Expert 1 compiled."
fi

ls -lh ${COMPILED_DIR}/transformer/nxd_model.pt

### 6c: Compile Transformer Expert 2 (Low Noise)

In [ ]:
%%bash -e
NVME_MOUNT="/opt/dlami/nvme"
HENAN_DIR="${NVME_MOUNT}/aws-neuron-samples/torch-neuronx/inference/hf_pretrained_wan2.2_t2v_a14b"
COMPILED_DIR="${NVME_MOUNT}/compiled_a14b_tp4"
CACHE_DIR="${NVME_MOUNT}/wan2.2_t2v_a14b_hf_cache_dir"
COMPILER_WORKDIR="${NVME_MOUNT}/compiler_workdir"
export PYTHONPATH="${HENAN_DIR}:${PYTHONPATH:-}"
export NEURON_RT_VIRTUAL_CORE_SIZE=2

if [ -f "${COMPILED_DIR}/transformer_2/nxd_model.pt" ]; then
    echo "Expert 2 already compiled"
else
    echo "Compiling transformer Expert 2 (TP=4, CP=16)..."
    WORLD_SIZE=64 neuron_parallel_compile python \
        ${HENAN_DIR}/neuron_wan2_2_t2v_a14b/compile_transformer.py \
        --height 768 --width 1280 --num_frames 81 --max_sequence_length 512 \
        --tp_degree 4 --cp_degree 16 --batch_size 1 \
        --transformer_subfolder transformer_2 \
        --compiled_models_dir ${COMPILED_DIR} \
        --compiler_workdir ${COMPILER_WORKDIR}/expert2 \
        --cache_dir ${CACHE_DIR} \
        2>&1 | tail -10
    echo "Expert 2 compiled."
fi

ls -lh ${COMPILED_DIR}/transformer_2/nxd_model.pt

### 6d: Compile Tiled VAE Decoder

In [ ]:
%%bash -e
NVME_MOUNT="/opt/dlami/nvme"
HENAN_DIR="${NVME_MOUNT}/aws-neuron-samples/torch-neuronx/inference/hf_pretrained_wan2.2_t2v_a14b"
COMPILED_DIR="${NVME_MOUNT}/compiled_a14b_tp4"
CACHE_DIR="${NVME_MOUNT}/wan2.2_t2v_a14b_hf_cache_dir"
COMPILER_WORKDIR="${NVME_MOUNT}/compiler_workdir"
export PYTHONPATH="${HENAN_DIR}:${PYTHONPATH:-}"
export NEURON_RT_VIRTUAL_CORE_SIZE=2

if [ -f "${COMPILED_DIR}/decoder_tile_ws1/nxd_model.pt" ]; then
    echo "Tiled VAE decoder already compiled"
else
    echo "Compiling tiled VAE decoder (416x416 tiles)..."
    NEURON_RT_NUM_CORES=2 python \
        ${HENAN_DIR}/neuron_wan2_2_t2v_a14b/compile_decoder_rolling.py \
        --height 416 --width 416 --num_frames 81 \
        --tp_degree 1 --world_size 1 \
        --output_subdir decoder_tile_ws1 \
        --compiled_models_dir ${COMPILED_DIR} \
        --compiler_workdir ${COMPILER_WORKDIR}/decoder \
        --cache_dir ${CACHE_DIR} \
        --skip_pqc \
        2>&1 | tail -10
    echo "Tiled VAE decoder compiled."
fi

ls -lh ${COMPILED_DIR}/decoder_tile_ws1/nxd_model.pt

### Verify All Compiled Artifacts

In [ ]:
import json

artifacts = {
    "text_encoder": f"{COMPILED_DIR}/text_encoder/nxd_model.pt",
    "transformer (Expert 1)": f"{COMPILED_DIR}/transformer/nxd_model.pt",
    "transformer_2 (Expert 2)": f"{COMPILED_DIR}/transformer_2/nxd_model.pt",
    "decoder_tile_ws1 (VAE)": f"{COMPILED_DIR}/decoder_tile_ws1/nxd_model.pt",
}

all_ok = True
for name, path in artifacts.items():
    if os.path.exists(path):
        size_mb = os.path.getsize(path) / 1024 / 1024
        print(f"  OK: {name} ({size_mb:.0f} MB)")
    else:
        print(f"  MISSING: {name}")
        all_ok = False

# Show transformer config
config_path = f"{COMPILED_DIR}/transformer/config.json"
if os.path.exists(config_path):
    with open(config_path) as f:
        config = json.load(f)
    print(f"\nTransformer config: TP={config['tp_degree']}, CP={config['cp_degree']}, "
          f"world_size={config['world_size']}, seq_len={config['seq_len']}")

assert all_ok, "Some artifacts are missing. Check compilation logs above."

## Step 7: Run Inference

Now we run the full inference pipeline. This uses subprocess-based execution to manage HBM lifecycle:

- **Expert 1** runs on NeuronCores 0-63 (16 high-noise steps, all 64 cores)
- **Expert 2** runs on NeuronCores 0-63 (34 low-noise steps, sequential after Expert 1)
- **VAE decoder** runs on cores 0-7 (8 parallel tiles)

The worker script (`worker_denoise.py`) must be in the same directory as this notebook.

**Expected time**: ~10 minutes (model loading + 50 denoising steps + VAE decode)

In [ ]:
# Verify worker scripts exist
notebook_dir = os.path.dirname(os.path.abspath("__file__"))

# When run via nbconvert, __file__ may not be set — use cwd
if not os.path.exists(os.path.join(notebook_dir, "worker_denoise.py")):
    notebook_dir = os.getcwd()

worker_script = os.path.join(notebook_dir, "worker_denoise.py")

assert os.path.exists(worker_script), f"Missing: {worker_script}"
print(f"Worker script found in: {notebook_dir}")

In [ ]:
import gc
import json
import math
import random
import shutil
import tempfile
import time

import numpy as np
import torch
import torch.nn.functional as F
from diffusers import AutoencoderKLWan, WanPipeline
from diffusers.utils import export_to_video

DTYPE = torch.bfloat16
SEED = 42
GUIDANCE_SCALE_HIGH = 4.0
GUIDANCE_SCALE_LOW = 3.0
BOUNDARY_RATIO = 0.875
MAX_SEQ_LEN = 512
TILED_DECODER_WORKER = "neuron_wan2_2_t2v_a14b.tiled_decoder_worker"

PROMPT = "A cat walks on the grass, realistic style, 4K quality"
NEGATIVE_PROMPT = (
    "Bright tones, overexposed, static, blurred details, subtitles, style, "
    "works, paintings, images, static, overall gray, worst quality, low quality, "
    "JPEG compression residue, ugly, incomplete, extra fingers, poorly drawn hands, "
    "poorly drawn faces, deformed, disfigured, misshapen limbs, fused fingers, "
    "still picture, messy background, three legs, many people in the background, "
    "walking backwards"
)

OUTPUT_PATH = "output_wan22_a14b.mp4"

print(f"Prompt: {PROMPT}")
print(f"Output: {OUTPUT_PATH}")

In [ ]:
# Helper functions

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)


def get_neuron_env(world_size, core_start):
    """Build Neuron RT environment dict for a subprocess."""
    return {
        "NEURON_RT_NUM_CORES": str(world_size),
        "NEURON_RT_VIRTUAL_CORE_SIZE": "2",
        "NEURON_RT_VISIBLE_CORES": f"{core_start}-{core_start + world_size - 1}",
        "NEURON_RT_INSPECT_ENABLE": "0",
        "NEURON_RT_INSPECT_DEVICE_PROFILE": "0",
        "NEURON_RT_INSPECT_SYSTEM_PROFILE": "0",
        "NEURON_RT_PROFILING_MODE": "0",
    }


def make_clean_env():
    """Make a clean subprocess env (no inherited NEURON_RT_ vars)."""
    env = os.environ.copy()
    for k in list(env.keys()):
        if k.startswith("NEURON_RT_") or k == "NEURON_LOGICAL_NC_CONFIG":
            del env[k]
    env["PYTHONUNBUFFERED"] = "1"
    return env

In [ ]:
# Expert subprocess functions

def run_expert_subprocess(
    compiled_path, pipe_transformer_state, latents,
    prompt_embeds, negative_prompt_embeds,
    step_start, step_end, guidance_scale, expand_timesteps, mask,
    scheduler_config, scheduler_state, num_inference_steps,
    world_size, core_start, label="expert",
):
    """Run denoising steps in a subprocess for clean HBM lifecycle."""
    tmpdir = tempfile.mkdtemp(prefix=f"denoise_{label}_")
    input_path = os.path.join(tmpdir, "input.pt")
    output_path = os.path.join(tmpdir, "output.pt")
    neuron_env = get_neuron_env(world_size, core_start)

    phase_data = {
        "compiled_path": compiled_path,
        "pipe_transformer_state": pipe_transformer_state,
        "latents": latents,
        "prompt_embeds": prompt_embeds,
        "negative_prompt_embeds": negative_prompt_embeds,
        "step_start": step_start, "step_end": step_end,
        "guidance_scale": guidance_scale,
        "expand_timesteps": expand_timesteps,
        "mask": mask,
        "scheduler_config": dict(scheduler_config),
        "scheduler_state": scheduler_state,
        "num_inference_steps": num_inference_steps,
        "neuron_env": neuron_env,
    }
    torch.save(phase_data, input_path)

    print(f"[Main] Launching {label} (steps {step_start}-{step_end-1}, "
          f"cores {core_start}-{core_start+world_size-1})...")
    t0 = time.time()
    env = make_clean_env()
    result = subprocess.run(
        [sys.executable, worker_script, input_path, output_path],
        env=env, timeout=3600,
    )
    if result.returncode != 0:
        raise RuntimeError(f"{label} failed with code {result.returncode}")

    output = torch.load(output_path, weights_only=False)
    elapsed = time.time() - t0
    print(f"[Main] {label} done in {elapsed:.1f}s "
          f"(load: {output['load_time']:.1f}s, denoise: {output['denoise_time']:.1f}s)")

    shutil.rmtree(tmpdir, ignore_errors=True)
    return output["latents"], output.get("scheduler_state"), output

In [ ]:
# Neuron tiled VAE decode

def vae_decode_neuron_tiled(pipe, compiled_models_dir, latents, num_frames, cwd):
    """Decode latents using parallel tiled Neuron VAE."""
    print("\nVAE Decoding (Neuron Parallel Tiled)")

    vae_config = pipe.vae.config
    latents = latents.to(torch.float32)
    latents_mean = torch.tensor(vae_config.latents_mean).view(1, vae_config.z_dim, 1, 1, 1)
    latents_std = 1.0 / torch.tensor(vae_config.latents_std).view(1, vae_config.z_dim, 1, 1, 1)
    latents = latents / latents_std + latents_mean
    print(f"Denormalized latents: {latents.shape}, range=[{latents.min():.3f}, {latents.max():.3f}]")

    pipe.vae.post_quant_conv.to(torch.float32)
    with torch.no_grad():
        z = pipe.vae.post_quant_conv(latents)
    z_bf16 = z.to(torch.bfloat16)
    del z, latents
    gc.collect()

    B, C, T, H_lat, W_lat = z_bf16.shape

    decoder_path = os.path.join(compiled_models_dir, "decoder_tile_ws1")
    with open(os.path.join(decoder_path, "config.json")) as f:
        decoder_config = json.load(f)
    tile_h = decoder_config["height"] // 8
    tile_w = decoder_config["width"] // 8
    decoder_frames = decoder_config.get("decoder_frames", 2)
    decoder_world_size = decoder_config["world_size"]

    # Compute tile grid with overlap
    def compute_starts(total, tile_size):
        if total <= tile_size:
            return [0]
        n = math.ceil(total / tile_size)
        stride = (total - tile_size) / max(n - 1, 1)
        return [round(i * stride) for i in range(n)]

    h_starts = compute_starts(H_lat, tile_h)
    w_starts = compute_starts(W_lat, tile_w)
    n_tiles = len(h_starts) * len(w_starts)
    print(f"Tiling: latent {H_lat}x{W_lat} -> {len(h_starts)}x{len(w_starts)} = {n_tiles} tiles")

    core_start = 0
    base_env = make_clean_env()
    if "PYTHONPATH" not in base_env:
        base_env["PYTHONPATH"] = cwd
    elif cwd not in base_env["PYTHONPATH"]:
        base_env["PYTHONPATH"] = cwd + ":" + base_env["PYTHONPATH"]

    tmpdir = tempfile.mkdtemp(prefix="tiled_dec_")
    decode_start = time.time()

    tile_info = []
    tile_idx = 0
    for hi, h_start in enumerate(h_starts):
        for wi, w_start in enumerate(w_starts):
            actual_h = min(tile_h, H_lat - h_start)
            actual_w = min(tile_w, W_lat - w_start)
            z_tile = z_bf16[:, :, :, h_start:h_start+actual_h, w_start:w_start+actual_w]
            if actual_h < tile_h or actual_w < tile_w:
                z_tile = F.pad(z_tile, (0, tile_w - actual_w, 0, tile_h - actual_h))

            tile_input = os.path.join(tmpdir, f"tile_{tile_idx}_input.pt")
            tile_output = os.path.join(tmpdir, f"tile_{tile_idx}_output.pt")
            tile_env_path = os.path.join(tmpdir, f"tile_{tile_idx}_env.json")

            torch.save({
                "tile_data": z_tile,
                "decoder_path": decoder_path,
                "decoder_frames": decoder_frames,
                "num_video_frames": num_frames,
                "tile_id": f"{hi},{wi}",
            }, tile_input)

            nc = core_start + tile_idx * decoder_world_size
            env_config = {
                "NEURON_RT_NUM_CORES": str(decoder_world_size),
                "NEURON_RT_VIRTUAL_CORE_SIZE": "2",
                "NEURON_RT_VISIBLE_CORES": f"{nc}-{nc + decoder_world_size - 1}",
                "NEURON_RT_INSPECT_ENABLE": "0",
                "NEURON_RT_INSPECT_DEVICE_PROFILE": "0",
                "NEURON_RT_INSPECT_SYSTEM_PROFILE": "0",
                "NEURON_RT_PROFILING_MODE": "0",
            }
            with open(tile_env_path, "w") as f:
                json.dump(env_config, f)

            proc = subprocess.Popen(
                [sys.executable, "-m", TILED_DECODER_WORKER, tile_input, tile_output, tile_env_path],
                cwd=cwd, env=base_env,
            )
            tile_info.append((hi, wi, h_start, w_start, actual_h, actual_w, proc, tile_output))
            print(f"  Launched tile ({hi},{wi}) on NC {nc}")
            tile_idx += 1

    del z_bf16
    gc.collect()

    print(f"Waiting for {n_tiles} tile subprocesses...")
    for hi, wi, _, _, _, _, proc, _ in tile_info:
        ret = proc.wait()
        if ret != 0:
            raise RuntimeError(f"Tile ({hi},{wi}) subprocess failed with code {ret}")
    print(f"All tiles done in {time.time() - decode_start:.1f}s")

    # Collect and blend with linear ramp overlap
    H_pix, W_pix = H_lat * 8, W_lat * 8
    output_acc = torch.zeros(3, num_frames, H_pix, W_pix, dtype=torch.float32)
    weight_acc = torch.zeros(H_pix, W_pix, dtype=torch.float32)
    max_decode_time = max_load_time = 0

    for hi, wi, h_start, w_start, actual_h, actual_w, _, tile_out_path in tile_info:
        result = torch.load(tile_out_path, weights_only=False)
        tile_video = result["tile_video"]
        max_decode_time = max(max_decode_time, result["decode_time"])
        max_load_time = max(max_load_time, result["load_time"])

        if tile_video.shape[2] > num_frames:
            tile_video = tile_video[:, :, :num_frames]

        ah_pix, aw_pix = actual_h * 8, actual_w * 8
        tile_video = tile_video[:, :, :, :ah_pix, :aw_pix]

        h_weight = torch.ones(ah_pix)
        w_weight = torch.ones(aw_pix)
        if hi > 0:
            overlap = (h_starts[hi-1] + tile_h - h_start) * 8
            if overlap > 0:
                ramp = min(overlap, ah_pix)
                h_weight[:ramp] = torch.linspace(0, 1, ramp + 2)[1:-1]
        if hi < len(h_starts) - 1:
            overlap = (h_start + tile_h - h_starts[hi+1]) * 8
            if overlap > 0:
                ramp = min(overlap, ah_pix)
                h_weight[-ramp:] = torch.linspace(1, 0, ramp + 2)[1:-1]
        if wi > 0:
            overlap = (w_starts[wi-1] + tile_w - w_start) * 8
            if overlap > 0:
                ramp = min(overlap, aw_pix)
                w_weight[:ramp] = torch.linspace(0, 1, ramp + 2)[1:-1]
        if wi < len(w_starts) - 1:
            overlap = (w_start + tile_w - w_starts[wi+1]) * 8
            if overlap > 0:
                ramp = min(overlap, aw_pix)
                w_weight[-ramp:] = torch.linspace(1, 0, ramp + 2)[1:-1]

        weight_2d = h_weight.unsqueeze(1) * w_weight.unsqueeze(0)
        hp, wp = h_start * 8, w_start * 8
        output_acc[:, :, hp:hp+ah_pix, wp:wp+aw_pix] += tile_video[0] * weight_2d.unsqueeze(0).unsqueeze(0)
        weight_acc[hp:hp+ah_pix, wp:wp+aw_pix] += weight_2d
        del tile_video, result

    video = output_acc / weight_acc.unsqueeze(0).unsqueeze(0).clamp(min=1e-6)
    decode_time = time.time() - decode_start
    print(f"Parallel tiled decode: {decode_time:.1f}s (max load: {max_load_time:.1f}s, max decode: {max_decode_time:.1f}s)")
    print(f"Output: {list(video.shape)}")

    del output_acc, weight_acc
    gc.collect()
    shutil.rmtree(tmpdir, ignore_errors=True)
    return video.unsqueeze(0), decode_time

In [ ]:
# ============================================================
# RUN INFERENCE
# ============================================================

total_start = time.time()
set_seed(SEED)

# --- Phase 1: Load pipeline + text encoding ---
print("=" * 60)
print("Phase 1: Pipeline Load + CPU Text Encoding")
print("=" * 60)

t0 = time.time()
vae = AutoencoderKLWan.from_pretrained(MODEL_DIR, subfolder="vae", torch_dtype=torch.float32)
pipe = WanPipeline.from_pretrained(MODEL_DIR, vae=vae, torch_dtype=DTYPE)
pipe_load_time = time.time() - t0
print(f"Pipeline loaded in {pipe_load_time:.1f}s")

t0 = time.time()
prompt_embeds, negative_prompt_embeds = pipe.encode_prompt(
    prompt=PROMPT,
    negative_prompt=NEGATIVE_PROMPT,
    do_classifier_free_guidance=True,
    num_videos_per_prompt=1,
    max_sequence_length=MAX_SEQ_LEN,
    device=torch.device("cpu"),
)
te_time = time.time() - t0
print(f"Text encoding: {prompt_embeds.shape} ({te_time:.1f}s)")
prompt_embeds = prompt_embeds.to(DTYPE)
negative_prompt_embeds = negative_prompt_embeds.to(DTYPE)

In [ ]:
# --- Phase 2: Prepare denoising ---
print("\n" + "=" * 60)
print("Phase 2: Prepare Denoising")
print("=" * 60)

device = torch.device("cpu")
pipe.scheduler.set_timesteps(NUM_STEPS, device=device)
timesteps = pipe.scheduler.timesteps

in_channels = pipe.transformer.config.in_channels if pipe.transformer is not None else 16
generator = torch.Generator().manual_seed(SEED)
latents = pipe.prepare_latents(1, in_channels, HEIGHT, WIDTH, NUM_FRAMES, torch.float32, device, generator, None)
mask = torch.ones(latents.shape, dtype=torch.float32, device=device)
print(f"Latents: {latents.shape}, timesteps: {len(timesteps)}")

# MoE boundary
boundary_timestep = BOUNDARY_RATIO * 1000
switch_idx = next((i for i, t in enumerate(timesteps) if t < boundary_timestep), len(timesteps))
remaining = len(timesteps) - switch_idx
print(f"MoE switch at step {switch_idx}: expert1={switch_idx} steps, expert2={remaining} steps")

# Read compiled config
with open(f"{COMPILED_DIR}/transformer/config.json") as f:
    config = json.load(f)
world_size = config["world_size"]
tp_degree = config["tp_degree"]
cp_degree = config["cp_degree"]
print(f"Compiled: TP={tp_degree}, CP={cp_degree}, world_size={world_size}")

scheduler_config = dict(pipe.scheduler.config)
scheduler_state = {attr: getattr(pipe.scheduler, attr, None)
                   for attr in ["order_list", "model_outputs", "timestep_list", "lower_order_nums", "sample"]}

# Capture transformer state dicts for norm weight fixing
pipe_t1_state = dict(pipe.transformer.state_dict())
pipe_t2_state = dict(pipe.transformer_2.state_dict())
del pipe.transformer, pipe.transformer_2
pipe.transformer = None
pipe.transformer_2 = None
gc.collect()

In [ ]:
# --- Phase 3: Expert 1 (high-noise) ---
print("\n" + "=" * 60)
print("Phase 3: Expert 1 (High Noise)")
print("=" * 60)

t1_path = f"{COMPILED_DIR}/transformer"
t2_path = f"{COMPILED_DIR}/transformer_2"

# Core assignment (TP=4/CP=16, world_size=64 — all cores)
core_start = 0  # cores 0-63

# Run Expert 1 (blocks until complete)
latents, scheduler_state, out1 = run_expert_subprocess(
    t1_path, pipe_t1_state, latents,
    prompt_embeds, negative_prompt_embeds,
    0, switch_idx, GUIDANCE_SCALE_HIGH,
    pipe.config.expand_timesteps, mask,
    scheduler_config, scheduler_state, NUM_STEPS,
    world_size, core_start, "expert1_high_noise",
)
del pipe_t1_state
gc.collect()

In [ ]:
# --- Phase 4: Expert 2 (low-noise) ---
print("\n" + "=" * 60)
print("Phase 4: Expert 2 (Low Noise)")
print("=" * 60)

latents, _, out2 = run_expert_subprocess(
    t2_path, pipe_t2_state, latents,
    prompt_embeds, negative_prompt_embeds,
    switch_idx, len(timesteps), GUIDANCE_SCALE_LOW,
    pipe.config.expand_timesteps, mask,
    scheduler_config, scheduler_state, NUM_STEPS,
    world_size, core_start, "expert2_low_noise",
)
del pipe_t2_state
gc.collect()

In [ ]:
# --- Phase 5: VAE Decode ---
print("\n" + "=" * 60)
print("Phase 5: VAE Decode (Neuron Tiled)")
print("=" * 60)

video, vae_time = vae_decode_neuron_tiled(pipe, COMPILED_DIR, latents, NUM_FRAMES, HENAN_DIR)

In [ ]:
# --- Phase 6: Export video ---
video_np = video[0].permute(1, 2, 3, 0).float().cpu().numpy()
video_np = ((video_np + 1.0) / 2.0).clip(0, 1)
frames = [video_np[i] for i in range(video_np.shape[0])]
export_to_video(frames, OUTPUT_PATH, fps=16)

total_time = time.time() - total_start
denoise_only = out1["denoise_time"] + out2["denoise_time"]
load_only = out1["load_time"] + out2["load_time"]

print(f"\n{'=' * 60}")
print(f"VIDEO SAVED: {OUTPUT_PATH} ({len(frames)} frames)")
print(f"{'=' * 60}")
print(f"  Resolution:              {HEIGHT}x{WIDTH}, {NUM_FRAMES}f")
print(f"  Config:                  TP={tp_degree}, CP={cp_degree}")
print(f"  Pipeline load:           {pipe_load_time:.1f}s")
print(f"  Text encoding (CPU):     {te_time:.1f}s")
print(f"  Expert 1 load:           {out1['load_time']:.1f}s")
print(f"  Expert 1 denoise:        {out1['denoise_time']:.1f}s ({switch_idx} steps)")
print(f"  Expert 2 load:           {out2['load_time']:.1f}s")
print(f"  Expert 2 denoise:        {out2['denoise_time']:.1f}s ({remaining} steps)")
print(f"  VAE decode (Neuron):     {vae_time:.1f}s")
print(f"  ---")
print(f"  Model loading total:     {load_only:.1f}s")
print(f"  Denoising total:         {denoise_only:.1f}s")
print(f"  Per-step avg:            {denoise_only / NUM_STEPS:.1f}s")
print(f"  Total wall time:         {total_time:.1f}s")
print(f"{'=' * 60}")

In [ ]:
# Display a sample frame
from IPython.display import Image as IPImage, display
from PIL import Image
import io

frame = frames[len(frames) // 2]  # middle frame
img = Image.fromarray((frame * 255).astype(np.uint8))
buf = io.BytesIO()
img.save(buf, format='PNG')
display(IPImage(data=buf.getvalue()))
print(f"Frame {len(frames)//2} of {len(frames)}")